# Task 1 — Clickbait Spoiler-Type Classification

This notebook documents the development and evaluation of several models for Task 1.

The experiments include:

- RoBERTa using `postText` only;
- learning-rate and calibration experiments;
- structured metadata using title and description;
- article-context modelling using `postText` and `targetParagraphs`.

The final selected model is RoBERTa-base using:

`postText </s></s> targetParagraphs`

Final Kaggle score: **0.78270**

In [16]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

In [17]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Device: cuda


## 1. Load the Data

The competition provides separate training, validation, and test files. The training and validation sets include clickbait labels, while the test set is used to generate the final Kaggle submission.

In [18]:
DATA_DIR = "/kaggle/input/competitions/task-1-clickbait-detection-mse-641-s-26"

train_df = pd.read_json(
    os.path.join(DATA_DIR, "train.jsonl"),
    lines=True
)

val_df = pd.read_json(
    os.path.join(DATA_DIR, "val.jsonl"),
    lines=True
)

test_df = pd.read_json(
    os.path.join(DATA_DIR, "test.jsonl"),
    lines=True
)

print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())

Training shape: (3200, 14)
Validation shape: (400, 14)
Test shape: (400, 10)


,uuid,postId,postText,postPlatform,targetParagraphs,targetTitle,targetDescription,targetKeywords,targetMedia,targetUrl,provenance,spoiler,spoilerPositions,tags
0,0af11f6b-c889-4520-9372-66ba25cb7657,532quh,"[Wes Welker Wanted Dinner With Tom Brady, But ...",reddit,[It’ll be just like old times this weekend for...,"Wes Welker Wanted Dinner With Tom Brady, But P...",It'll be just like old times this weekend for ...,"new england patriots, ricky doyle, top stories,","[http://pixel.wp.com/b.gif?v=noscript, http://...",http://nesn.com/2016/09/wes-welker-wanted-dinn...,"{'source': 'anonymized', 'humanSpoiler': 'They...",[how about that morning we go throw?],"[[[3, 151], [3, 186]]]",[passage]
1,b1a1f63d-8853-4a11-89e8-6b2952a393ec,411701128456593408,[NASA sets date for full recovery of ozone hole],Twitter,[2070 is shaping up to be a great year for Mot...,Hole In Ozone Layer Expected To Make Full Reco...,2070 is shaping up to be a great year for Moth...,"ozone layer,ozone hole determined by weather,M...",[http://s.m.huffpost.com/assets/Logo_Huffingto...,http://huff.to/1cH672Z,"{'source': 'anonymized', 'humanSpoiler': '2070...",[2070],"[[[0, 0], [0, 4]]]",[phrase]
2,008b7b19-0445-4e16-8f9e-075b73f80ca4,380537005123190784,[This is what makes employees happy -- and it'...,Twitter,"[Despite common belief, money isn't the key to...",Intellectual Stimulation Trumps Money For Empl...,By: Chad Brooks \r\nPublished: 09/18/2013 06:4...,"employee happiness money,employee happiness in...",[http://i.huffpost.com/gen/1359674/images/o-HA...,http://huff.to/1epfeaw,"{'source': 'anonymized', 'humanSpoiler': 'Inte...",[intellectual stimulation],"[[[1, 186], [1, 210]]]",[phrase]
3,31ecf93c-3e21-4c80-949b-aa549a046b93,844567852531286016,[Passion is overrated — 7 work habits you need...,Twitter,"[It’s common wisdom. Near gospel really, and n...","‘Follow your passion’ is wrong, here are 7 hab...",There's a lot more to work that loving your job,"business, work-life, careers",None,None,"{'source': 'anonymized', 'humanSpoiler': None,...",[Purpose connects us to something bigger and i...,"[[[11, 25], [11, 101]], [[17, 56], [17, 85]], ...",[multi]
4,31b108a3-c828-421a-a4b9-cf651e9ac859,814186311573766144,[The perfect way to cook rice so that it's per...,Twitter,"[Boiling rice may seem simple, but there is a ...",Revealed: The perfect way to cook rice so that...,The question 'How does one cook rice properly?...,"Quora,users,share,perfect,way,cook,rice",None,None,"{'source': 'anonymized', 'humanSpoiler': None,...",[in a rice cooker],"[[[5, 60], [5, 76]]]",[phrase]


## 2. Prepare Labels and Text

The target label is stored in the `tags` column as a one-item list.

The initial baseline experiments use only `postText`. Later experiments add article metadata and article paragraphs to evaluate whether additional context improves hidden-test performance.

In [19]:
label2id = {
    "phrase": 0,
    "passage": 1,
    "multi": 2
}

id2label = {
    0: "phrase",
    1: "passage",
    2: "multi"
}

for df in [train_df, val_df]:
    df["label"] = df["tags"].apply(lambda x: label2id[x[0]])

for df in [train_df, val_df, test_df]:
    df["post_text"] = df["postText"].apply(
        lambda x: " ".join(x) if isinstance(x, list) else str(x)
    )

print("Training label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(val_df["label"].value_counts().sort_index())

display(train_df[["postText", "post_text", "tags", "label"]].head())

Training label distribution:
label
0    1367
1    1274
2     559
Name: count, dtype: int64

Validation label distribution:
label
0    162
1    154
2     84
Name: count, dtype: int64


,postText,post_text,tags,label
0,"[Wes Welker Wanted Dinner With Tom Brady, But ...","Wes Welker Wanted Dinner With Tom Brady, But P...",[passage],1
1,[NASA sets date for full recovery of ozone hole],NASA sets date for full recovery of ozone hole,[phrase],0
2,[This is what makes employees happy -- and it'...,This is what makes employees happy -- and it's...,[phrase],0
3,[Passion is overrated — 7 work habits you need...,Passion is overrated — 7 work habits you need ...,[multi],2
4,[The perfect way to cook rice so that it's per...,The perfect way to cook rice so that it's perf...,[phrase],0


## 3. Create RoBERTa Datasets

The pandas DataFrames are converted into Hugging Face datasets.  
The `post_text` field is tokenized using the RoBERTa tokenizer with truncation and a fixed maximum sequence length.

In [20]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(
    train_df[["post_text", "label"]].reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df[["post_text", "label"]].reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df[["post_text"]].reset_index(drop=True)
)

def tokenize_batch(batch):
    return tokenizer(
        batch["post_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["post_text"]
)

val_dataset = val_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["post_text"]
)

test_dataset = test_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["post_text"]
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 3200
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 400
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 400
})


## 4. Define Evaluation Metrics

The competition is evaluated using macro F1, so macro F1 is the main model-selection metric. Accuracy and weighted F1 are also reported for additional context.

In [21]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted")
    }

## 5. Initial RoBERTa Baseline

The first controlled transformer experiment uses RoBERTa-base with `postText` only.

The training configuration uses:

- seed 42;
- learning rate `1e-5`;
- up to 5 epochs;
- early stopping;
- macro F1 for selecting the best checkpoint.

This model serves as an initial transformer baseline rather than the final selected model.

In [22]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

training_args = TrainingArguments(
    output_dir="/kaggle/working/roberta_task1_seed42",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    seed=SEED,
    data_seed=SEED,

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

print("Trainer is ready.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer is ready.


## 6. Training

The model is fine-tuned on the training set. After each epoch, performance is evaluated on the validation set. The checkpoint with the highest validation macro F1 is retained.

In [23]:
train_result = trainer.train()

print("Training completed.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro F1:", trainer.state.best_metric)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,2.042165,1.899267,0.537500,0.398562,0.472770
2,1.666516,1.480552,0.700000,0.684185,0.694479
3,1.434416,1.402547,0.732500,0.719900,0.729818
4,1.293809,1.432273,0.692500,0.683954,0.689398
5,1.208127,1.392403,0.730000,0.718395,0.727957


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training completed.
Best checkpoint: /kaggle/working/roberta_task1_seed42/checkpoint-300
Best validation macro F1: 0.7199002407980736


## Seed and Reproducibility Note

Seed 42 was used consistently across the main experiments to improve reproducibility.

Repeated transformer fine-tuning still produced some variation across runs. Therefore, model selection was based on both validation performance and Kaggle leaderboard results rather than repeatedly searching for a favorable random seed.

## 7. Evaluate the Best Checkpoint

The trainer automatically reloads the checkpoint with the highest validation macro F1. This section reports the final validation metrics and class-level performance.

In [24]:
eval_results = trainer.evaluate()

print("Final validation results:")
for metric, value in eval_results.items():
    print(f"{metric}: {value}")

val_output = trainer.predict(val_dataset)
val_predictions = np.argmax(val_output.predictions, axis=-1)
val_labels = np.array(val_dataset["label"])

print("\nClassification report:")
print(
    classification_report(
        val_labels,
        val_predictions,
        target_names=["phrase", "passage", "multi"],
        digits=4
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Final validation results:
eval_loss: 1.4024412631988525
eval_accuracy: 0.735
eval_macro_f1: 0.7219058267955184
eval_weighted_f1: 0.7322087472584182
eval_runtime: 0.7166
eval_samples_per_second: 558.182
eval_steps_per_second: 9.768
epoch: 5.0


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Classification report:
              precision    recall  f1-score   support

      phrase     0.7257    0.7840    0.7537       162
     passage     0.7143    0.7792    0.7453       154
       multi     0.8246    0.5595    0.6667        84

    accuracy                         0.7350       400
   macro avg     0.7549    0.7076    0.7219       400
weighted avg     0.7421    0.7350    0.7322       400



## Initial RoBERTa Validation Summary

The postText-only RoBERTa model with learning rate `1e-5` achieved:

- Accuracy: 0.7350
- Macro F1: 0.7219
- Weighted F1: 0.7322

This model improved on the earlier classical and transformer baselines, but its Kaggle score of 0.72513 was below later RoBERTa configurations. It was therefore retained as a baseline rather than selected as the final model.

In [25]:
## 8. Generate Kaggle Submission

sample_solution = pd.read_csv(
    os.path.join(DATA_DIR, "sample_solution.csv")
)

test_output = trainer.predict(test_dataset)
test_predictions = np.argmax(
    test_output.predictions,
    axis=-1
)

predicted_tags = [
    id2label[int(pred)]
    for pred in test_predictions
]

submission = sample_solution.copy()
submission["spoilerType"] = predicted_tags

submission_path = "/kaggle/working/submission_roberta_seed42_lr1e5.csv"
submission.to_csv(submission_path, index=False)

print("Submission shape:", submission.shape)
print("Submission columns:", submission.columns.tolist())

print("\nPrediction distribution:")
print(submission["spoilerType"].value_counts())

display(submission.head(10))

print("\nSaved to:", submission_path)

Submission shape: (400, 2)
Submission columns: ['id', 'spoilerType']

Prediction distribution:
spoilerType
passage    189
phrase     166
multi       45
Name: count, dtype: int64


,id,spoilerType
0,0,passage
1,1,passage
2,2,phrase
3,3,phrase
4,4,passage
5,5,phrase
6,6,multi
7,7,passage
8,8,phrase
9,9,multi



Saved to: /kaggle/working/submission_roberta_seed42_lr1e5.csv


## Kaggle Result

The seed-42 RoBERTa model with learning rate 1e-5 achieved:

- Validation macro F1: 0.7219
- Kaggle score: 0.72513

Although the local validation score improved, the Kaggle score was lower than the previous best score of 0.73686. Therefore, this configuration was rejected.

This suggests that the lower learning rate improved performance on the local validation set but did not generalize as well to the hidden Kaggle test set.

In [26]:
model_2e5 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

training_args_2e5 = TrainingArguments(
    output_dir="/kaggle/working/roberta_task1_seed42_lr2e5",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    seed=42,
    data_seed=42,
    report_to="none"
)

trainer_2e5 = Trainer(
    model=model_2e5,
    args=training_args_2e5,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

print("Winning-configuration trainer is ready.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Winning-configuration trainer is ready.


In [27]:
trainer_2e5.train()

print("Best checkpoint:", trainer_2e5.state.best_model_checkpoint)
print("Best validation macro F1:", trainer_2e5.state.best_metric)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.829260,1.451731,0.710000,0.694507,0.706277
2,1.433074,1.310434,0.747500,0.731376,0.743530
3,1.190042,1.360319,0.752500,0.742424,0.750394
4,0.959520,1.411626,0.745000,0.734573,0.743480
5,0.835986,1.430090,0.742500,0.733706,0.741386


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Best checkpoint: /kaggle/working/roberta_task1_seed42_lr2e5/checkpoint-300
Best validation macro F1: 0.7424242424242425


In [28]:
test_output_2e5 = trainer_2e5.predict(test_dataset)

test_predictions_2e5 = np.argmax(
    test_output_2e5.predictions,
    axis=-1
)

predicted_tags_2e5 = [
    id2label[int(pred)]
    for pred in test_predictions_2e5
]

submission_2e5 = sample_solution.copy()
submission_2e5["spoilerType"] = predicted_tags_2e5

submission_path_2e5 = (
    "/kaggle/working/submission_roberta_seed42_lr2e5.csv"
)

submission_2e5.to_csv(
    submission_path_2e5,
    index=False
)

print("Submission shape:", submission_2e5.shape)
print("\nPrediction distribution:")
print(submission_2e5["spoilerType"].value_counts())
print("\nSaved to:", submission_path_2e5)

display(submission_2e5.head())

Submission shape: (400, 2)

Prediction distribution:
spoilerType
passage    186
phrase     171
multi       43
Name: count, dtype: int64

Saved to: /kaggle/working/submission_roberta_seed42_lr2e5.csv


,id,spoilerType
0,0,phrase
1,1,passage
2,2,phrase
3,3,phrase
4,4,passage


## Best Result So Far

The strongest Kaggle result achieved so far used:

- Model: RoBERTa-base
- Input: `postText` only
- Seed: 42
- Learning rate: `2e-5`
- Best epoch: 3
- Validation macro F1 from that run: 0.7274
- Kaggle score: 0.74950

A later clean rerun of the same configuration achieved a higher validation macro F1 of 0.7424, but its Kaggle score was only 0.71760. This shows that repeated transformer fine-tuning can produce different trained models even under the same general configuration, and that higher local validation performance does not always correspond to better public-leaderboard performance.

Therefore, 0.74950 remains the best historical Kaggle result, while the current in-memory model is the clean rerun.

## Multi-Class Logit Calibration

The current model performs well overall but has lower recall for the `multi` class. This experiment adds a small positive bias to the `multi` logit and selects the bias that produces the highest validation macro F1.

The model weights are unchanged; only the final decision rule is adjusted.

In [29]:
val_output_2e5 = trainer_2e5.predict(val_dataset)

val_logits_2e5 = val_output_2e5.predictions
val_labels_2e5 = np.array(val_dataset["label"])

calibration_results = []

for multi_bias in np.arange(0.0, 1.05, 0.05):
    adjusted_logits = val_logits_2e5.copy()
    adjusted_logits[:, 2] += multi_bias

    adjusted_predictions = np.argmax(
        adjusted_logits,
        axis=-1
    )

    macro_f1 = f1_score(
        val_labels_2e5,
        adjusted_predictions,
        average="macro"
    )

    multi_recall = (
        adjusted_predictions[val_labels_2e5 == 2] == 2
    ).mean()

    calibration_results.append({
        "multi_bias": round(float(multi_bias), 2),
        "macro_f1": macro_f1,
        "multi_recall": multi_recall
    })

calibration_df = pd.DataFrame(calibration_results)

display(
    calibration_df.sort_values(
        "macro_f1",
        ascending=False
    ).head(10)
)

best_row = calibration_df.loc[
    calibration_df["macro_f1"].idxmax()
]

BEST_MULTI_BIAS = float(best_row["multi_bias"])

print("Best multi bias:", BEST_MULTI_BIAS)
print("Best adjusted macro F1:", best_row["macro_f1"])
print("Adjusted multi recall:", best_row["multi_recall"])

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


,multi_bias,macro_f1,multi_recall
4,0.20,0.750067,0.607143
6,0.30,0.749164,0.619048
7,0.35,0.746284,0.619048
8,0.40,0.746284,0.619048
2,0.10,0.746265,0.595238
1,0.05,0.746265,0.595238
3,0.15,0.746265,0.595238
11,0.55,0.745474,0.619048
9,0.45,0.745474,0.619048
10,0.50,0.745474,0.619048


Best multi bias: 0.2
Best adjusted macro F1: 0.7500673169343729
Adjusted multi recall: 0.6071428571428571


## Logit Calibration Result

For the current clean rerun, adding a bias of 0.20 to the `multi` class improved validation macro F1 from approximately 0.7424 to 0.7501.

The adjusted `multi` recall was 0.6071. Unlike the earlier calibration attempt, this rerun showed a measurable improvement, so a calibrated Kaggle submission was generated for evaluation.

Because the underlying clean rerun achieved only 0.71760 on Kaggle without calibration, the calibrated result is treated as an experimental submission rather than a replacement for the existing 0.74950 benchmark.

In [31]:
BEST_MULTI_BIAS = 0.20

test_output_calibrated = trainer_2e5.predict(test_dataset)
test_logits_calibrated = test_output_calibrated.predictions.copy()

# Add the selected validation-based bias to the multi class
test_logits_calibrated[:, 2] += BEST_MULTI_BIAS

test_predictions_calibrated = np.argmax(
    test_logits_calibrated,
    axis=-1
)

submission_calibrated = sample_solution.copy()

submission_calibrated["spoilerType"] = [
    id2label[int(pred)]
    for pred in test_predictions_calibrated
]

calibrated_path = (
    "/kaggle/working/submission_roberta_lr2e5_multi_bias020.csv"
)

submission_calibrated.to_csv(
    calibrated_path,
    index=False
)

print("Prediction distribution:")
print(submission_calibrated["spoilerType"].value_counts())

print("\nSaved to:", calibrated_path)
display(submission_calibrated.head())

Prediction distribution:
spoilerType
passage    184
phrase     169
multi       47
Name: count, dtype: int64

Saved to: /kaggle/working/submission_roberta_lr2e5_multi_bias020.csv


,id,spoilerType
0,0,phrase
1,1,passage
2,2,phrase
3,3,phrase
4,4,passage


## Calibrated Kaggle Result

Applying a `multi` logit bias of 0.20 improved the clean rerun's Kaggle score from 0.71760 to 0.72040.

Although calibration produced a small improvement of 0.00280, the result remained below the best historical score of 0.74950. Therefore, the calibrated model was rejected as the final submission.

This indicates that post-processing can improve a model slightly, but it cannot compensate for a weaker underlying trained instance.

## Structured Context Experiment

The current best model uses only `postText`. This experiment adds the linked article title and description as structured context.

Input format:

`postText </s></s> targetTitle </s></s> targetDescription`

All other training settings remain unchanged so that the effect of the additional context can be evaluated directly.

In [32]:
def clean_text_field(value):
    if value is None:
        return ""

    if isinstance(value, list):
        return " ".join(str(item) for item in value)

    return str(value)


for df in [train_df, val_df, test_df]:
    df["structured_text"] = (
        df["postText"].apply(clean_text_field)
        + " </s></s> "
        + df["targetTitle"].apply(clean_text_field)
        + " </s></s> "
        + df["targetDescription"].apply(clean_text_field)
    )

display(
    train_df[
        ["postText", "targetTitle", "targetDescription", "structured_text"]
    ].head(3)
)

,postText,targetTitle,targetDescription,structured_text
0,"[Wes Welker Wanted Dinner With Tom Brady, But ...","Wes Welker Wanted Dinner With Tom Brady, But P...",It'll be just like old times this weekend for ...,"Wes Welker Wanted Dinner With Tom Brady, But P..."
1,[NASA sets date for full recovery of ozone hole],Hole In Ozone Layer Expected To Make Full Reco...,2070 is shaping up to be a great year for Moth...,NASA sets date for full recovery of ozone hole...
2,[This is what makes employees happy -- and it'...,Intellectual Stimulation Trumps Money For Empl...,By: Chad Brooks \r\nPublished: 09/18/2013 06:4...,This is what makes employees happy -- and it's...


In [33]:
structured_train_dataset = Dataset.from_pandas(
    train_df[["structured_text", "label"]].reset_index(drop=True)
)

structured_val_dataset = Dataset.from_pandas(
    val_df[["structured_text", "label"]].reset_index(drop=True)
)

structured_test_dataset = Dataset.from_pandas(
    test_df[["structured_text"]].reset_index(drop=True)
)

def tokenize_structured(batch):
    return tokenizer(
        batch["structured_text"],
        truncation=True,
        max_length=192
    )

structured_train_dataset = structured_train_dataset.map(
    tokenize_structured,
    batched=True,
    remove_columns=["structured_text"]
)

structured_val_dataset = structured_val_dataset.map(
    tokenize_structured,
    batched=True,
    remove_columns=["structured_text"]
)

structured_test_dataset = structured_test_dataset.map(
    tokenize_structured,
    batched=True,
    remove_columns=["structured_text"]
)

print(structured_train_dataset)
print(structured_val_dataset)
print(structured_test_dataset)

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 3200
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 400
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 400
})


## Structured Context Model Training

A new RoBERTa model is trained using `postText`, `targetTitle`, and `targetDescription`. The training configuration is kept consistent with the current benchmark so that the effect of the added context can be evaluated.

In [34]:
# Reset randomness immediately before creating this model
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
set_seed(42)

structured_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

structured_args = TrainingArguments(
    output_dir="/kaggle/working/roberta_structured_context_run1",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    seed=42,
    data_seed=42,
    report_to="none"
)

structured_trainer = Trainer(
    model=structured_model,
    args=structured_args,
    train_dataset=structured_train_dataset,
    eval_dataset=structured_val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

print("Structured-context trainer is ready.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Structured-context trainer is ready.


In [35]:
structured_trainer.train()

print(
    "Best checkpoint:",
    structured_trainer.state.best_model_checkpoint
)

print(
    "Best validation macro F1:",
    structured_trainer.state.best_metric
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.999587,1.552842,0.667500,0.654258,0.660964
2,1.524423,1.399206,0.727500,0.709788,0.723031
3,1.225466,1.429611,0.722500,0.710525,0.719531
4,0.977347,1.500690,0.727500,0.718798,0.726028
5,0.783128,1.566141,0.727500,0.718686,0.725689


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Best checkpoint: /kaggle/working/roberta_structured_context_run1/checkpoint-400
Best validation macro F1: 0.7187979268067838


## Structured Context Result

Adding `targetTitle` and `targetDescription` to `postText` did not improve validation performance.

The model achieved its highest validation macro F1 of 0.7188 at epoch 4, which was below both the historical postText-only result of 0.7274 and the clean rerun result of 0.7424.

Therefore, the structured-context input was rejected. The additional article metadata likely introduced noise or redundant information that reduced the model's ability to focus on the linguistic structure of the clickbait post.

## PostText and Article-Body Experiment

The title and description experiment did not improve validation performance. This experiment instead combines the clickbait post with a preview of the linked article body.

Input format:

`postText </s></s> targetParagraphs`

Only the beginning of the article is retained through token truncation. All training settings remain consistent with the postText-only benchmark.

In [36]:
def join_paragraphs(value):
    if value is None:
        return ""

    if isinstance(value, list):
        return " ".join(
            str(paragraph)
            for paragraph in value
            if paragraph is not None
        )

    return str(value)


for df in [train_df, val_df, test_df]:
    df["post_article_text"] = (
        df["postText"].apply(clean_text_field)
        + " </s></s> "
        + df["targetParagraphs"].apply(join_paragraphs)
    )

display(
    train_df[
        ["postText", "targetParagraphs", "post_article_text"]
    ].head(3)
)

,postText,targetParagraphs,post_article_text
0,"[Wes Welker Wanted Dinner With Tom Brady, But ...",[It’ll be just like old times this weekend for...,"Wes Welker Wanted Dinner With Tom Brady, But P..."
1,[NASA sets date for full recovery of ozone hole],[2070 is shaping up to be a great year for Mot...,NASA sets date for full recovery of ozone hole...
2,[This is what makes employees happy -- and it'...,"[Despite common belief, money isn't the key to...",This is what makes employees happy -- and it's...


In [37]:
article_train_dataset = Dataset.from_pandas(
    train_df[["post_article_text", "label"]].reset_index(drop=True)
)

article_val_dataset = Dataset.from_pandas(
    val_df[["post_article_text", "label"]].reset_index(drop=True)
)

article_test_dataset = Dataset.from_pandas(
    test_df[["post_article_text"]].reset_index(drop=True)
)


def tokenize_post_article(batch):
    return tokenizer(
        batch["post_article_text"],
        truncation=True,
        max_length=256
    )


article_train_dataset = article_train_dataset.map(
    tokenize_post_article,
    batched=True,
    remove_columns=["post_article_text"]
)

article_val_dataset = article_val_dataset.map(
    tokenize_post_article,
    batched=True,
    remove_columns=["post_article_text"]
)

article_test_dataset = article_test_dataset.map(
    tokenize_post_article,
    batched=True,
    remove_columns=["post_article_text"]
)

print(article_train_dataset)
print(article_val_dataset)
print(article_test_dataset)

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 3200
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 400
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 400
})


In [38]:
# Reset randomness immediately before model creation
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
set_seed(42)

article_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

article_args = TrainingArguments(
    output_dir="/kaggle/working/roberta_post_article_run1",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    seed=42,
    data_seed=42,
    report_to="none"
)

article_trainer = Trainer(
    model=article_model,
    args=article_args,
    train_dataset=article_train_dataset,
    eval_dataset=article_val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

print("Article-body trainer is ready.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Article-body trainer is ready.


In [39]:
article_trainer.train()

print(
    "Best checkpoint:",
    article_trainer.state.best_model_checkpoint
)

print(
    "Best validation macro F1:",
    article_trainer.state.best_metric
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,2.052491,2.004435,0.465000,0.371662,0.421079
2,1.788136,1.592989,0.657500,0.656018,0.657198
3,1.473148,1.475057,0.700000,0.688969,0.697094
4,1.215813,1.461179,0.707500,0.698379,0.704045
5,1.042821,1.470657,0.725000,0.716721,0.722940


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Best checkpoint: /kaggle/working/roberta_post_article_run1/checkpoint-500
Best validation macro F1: 0.7167207861884796


In [40]:
article_test_output = article_trainer.predict(article_test_dataset)

article_test_predictions = np.argmax(
    article_test_output.predictions,
    axis=-1
)

article_submission = sample_solution.copy()

article_submission["spoilerType"] = [
    id2label[int(pred)]
    for pred in article_test_predictions
]

article_submission_path = (
    "/kaggle/working/submission_roberta_post_article.csv"
)

article_submission.to_csv(
    article_submission_path,
    index=False
)

print(article_submission["spoilerType"].value_counts())
print("Saved to:", article_submission_path)

spoilerType
passage    184
phrase     161
multi       55
Name: count, dtype: int64
Saved to: /kaggle/working/submission_roberta_post_article.csv


## Best Result So Far

The strongest Kaggle result achieved so far is:

- Model: RoBERTa-base
- Input: `postText </s></s> targetParagraphs`
- Seed: 42
- Learning rate: `2e-5`
- Max length: 256
- Best validation macro F1: 0.7167
- Kaggle score: 0.78270

Interestingly, this model underperformed on local validation compared with the earlier `postText`-only model, but substantially outperformed it on the Kaggle leaderboard. This suggests that the hidden test set is better aligned with article-body context than the local validation split.

## Article-Body Logit Calibration

The article-body model achieved the strongest Kaggle score of 0.78270. This experiment tests whether a small class-logit adjustment can improve its decision boundary without retraining the model.

The bias is selected using validation macro F1, and the original 0.78270 submission remains the protected benchmark.

In [41]:
article_val_output = article_trainer.predict(article_val_dataset)

article_val_logits = article_val_output.predictions
article_val_labels = np.array(article_val_dataset["label"])

article_calibration_results = []

for multi_bias in np.arange(-0.30, 0.35, 0.05):
    adjusted_logits = article_val_logits.copy()
    adjusted_logits[:, 2] += multi_bias

    adjusted_predictions = np.argmax(
        adjusted_logits,
        axis=-1
    )

    macro_f1 = f1_score(
        article_val_labels,
        adjusted_predictions,
        average="macro"
    )

    article_calibration_results.append({
        "multi_bias": round(float(multi_bias), 2),
        "macro_f1": macro_f1
    })

article_calibration_df = pd.DataFrame(
    article_calibration_results
)

display(
    article_calibration_df.sort_values(
        "macro_f1",
        ascending=False
    ).head(10)
)

best_article_row = article_calibration_df.loc[
    article_calibration_df["macro_f1"].idxmax()
]

BEST_ARTICLE_BIAS = float(
    best_article_row["multi_bias"]
)

print("Best article multi bias:", BEST_ARTICLE_BIAS)
print(
    "Best calibrated validation macro F1:",
    best_article_row["macro_f1"]
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


,multi_bias,macro_f1
0,-0.30,0.725568
1,-0.25,0.725568
2,-0.20,0.725568
3,-0.15,0.722722
4,-0.10,0.719714
5,-0.05,0.719714
7,0.05,0.717542
6,-0.00,0.716721
8,0.10,0.714562
9,0.15,0.714562


Best article multi bias: -0.3
Best calibrated validation macro F1: 0.7255680753768999


In [42]:
article_test_output = article_trainer.predict(article_test_dataset)

article_test_logits = article_test_output.predictions.copy()

# Apply the selected calibration only to the multi class
article_test_logits[:, 2] += BEST_ARTICLE_BIAS

article_calibrated_predictions = np.argmax(
    article_test_logits,
    axis=-1
)

article_calibrated_submission = sample_solution.copy()

article_calibrated_submission["spoilerType"] = [
    id2label[int(pred)]
    for pred in article_calibrated_predictions
]

article_calibrated_path = (
    "/kaggle/working/"
    "submission_roberta_post_article_calibrated_bias_minus_030.csv"
)

article_calibrated_submission.to_csv(
    article_calibrated_path,
    index=False
)

print(article_calibrated_submission["spoilerType"].value_counts())
print("Saved to:", article_calibrated_path)

spoilerType
passage    186
phrase     163
multi       51
Name: count, dtype: int64
Saved to: /kaggle/working/submission_roberta_post_article_calibrated_bias_minus_030.csv


# Final Experiment Summary

| Approach | Input | Validation Macro F1 | Kaggle Score | Decision |
|---|---|---:|---:|---|
| RoBERTa, learning rate `1e-5` | `postText` | 0.7219 | 0.72513 | Baseline |
| RoBERTa, learning rate `2e-5` | `postText` | 0.7274 | 0.74950 | Previous best |
| Clean rerun with calibration | `postText` | 0.7501 after calibration | 0.72040 | Rejected |
| Structured context | `postText + title + description` | 0.7188 | Not submitted | Rejected |
| Article context | `postText + targetParagraphs` | 0.7167 | **0.78270** | Final model |
| Calibrated article context | `postText + targetParagraphs` | 0.7256 | 0.77389 | Secondary result |

# Task 1 Final Conclusion

The final selected model is RoBERTa-base using the clickbait post together with the linked article paragraphs:

`postText </s></s> targetParagraphs`

The model used a learning rate of `2e-5`, seed 42, and a maximum sequence length of 256 tokens. It achieved a validation macro F1 of 0.7167 and a Kaggle leaderboard score of 0.78270.

Although its local validation macro F1 was lower than several postText-only models, it achieved the strongest hidden-test performance. This suggests that the local validation split was not fully representative of the Kaggle test distribution and that article-body context provided useful information for spoiler-type classification.

A calibrated version improved local validation macro F1 to 0.7256 but reduced the Kaggle score to 0.77389. Therefore, the uncalibrated article-context model was selected as the final Task 1 submission.